# Phase 4a: Contrastive Pair Construction (Candidate Generation)

Αυτό το notebook αναλαμβάνει το πρώτο μισό του Phase 4. Θα διαβάσει τα 318 missing features που φιλτράραμε και θα χρησιμοποιήσει το **Llama-3.1-8B-Instruct** για να δημιουργήσει (generate) υποψήφια τοξικά queries.

### ⚠️ ΣΗΜΑΝΤΙΚΟ: Hugging Face Token
Επειδή το τρέχεις πρώτη φορά στον λογαριασμό σου, πρέπει να κάνεις τα εξής:
1. Φτιάξε λογαριασμό στο [Hugging Face](https://huggingface.co/).
2. Πήγαινε στη σελίδα του [Llama-3.1-8B-Instruct](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct) και πάτα αποδοχή των όρων χρήσης (συνήθως δίνουν έγκριση άμεσα).
3. Πήγαινε στα [Settings > Access Tokens](https://huggingface.co/settings/tokens) και φτιάξε ένα νέο Token (τύπου Read).
4. Στο μενού αριστερά στο Colab, πάτα το εικονίδιο με το κλειδί (Secrets), φτιάξε ένα νέο secret με όνομα **`HF_TOKEN`** και κάνε επικόλληση το token σου (επίλεξε το Notebook access ενεργό).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Επιβεβαίωση ότι έχουμε GPU (T4 ή L4)
!nvidia-smi

## 1. Εγκατάσταση Βιβλιοθηκών

In [ ]:
!pip install -q transformers==4.43.4 accelerate==0.33.0 bitsandbytes datasets

## 2. Σύνδεση με Hugging Face

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# Θα τραβήξει αυτόματα το κλειδί που έβαλες στα Secrets του Colab
hf_token = userdata.get("HF_TOKEN")
login(hf_token)

## 3. Λήψη Κώδικα (FAC-Synthesis)

In [ ]:
%%bash
git clone https://github.com/Zhongzhi660/FAC-Synthesis.git

## 4. Patch για 4-bit Llama Loading
Το Llama 3.1 8B κανονικά απαιτεί 16GB VRAM. Ανάλογα με τη GPU που σου έδωσε το Colab (π.χ. T4 έχει 15GB), μπορεί να краσάρει (Out Of Memory). Για να είμαστε 100% σίγουροι, πατσάρουμε το `llama_wrapper.py` για να το φορτώσει σε 4-bit (θέλει μόνο ~6GB VRAM), όπως κάνατε και στο Phase 2!

In [ ]:
import os

wrapper_path = "/content/FAC-Synthesis/fac_synthesis/step1_contrastive_pair_construction/llama_wrapper.py"

with open(wrapper_path, "r") as f:
    code = f.read()

patch = """from transformers import BitsAndBytesConfig
import torch
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)"""

code = code.replace("""model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)""", patch)

with open(wrapper_path, "w") as f:
    f.write(code)

print("✅ Το llama_wrapper.py ενημερώθηκε επιτυχώς για 4-bit precision!")

## 5. Αντιγραφή του TSV με τα Missing Features
Πρέπει να αντιγράψεις το αρχείο `intersection_tox7_corr3.tsv` (που έχει τα 318 features) από το Google Drive σου στο Colab. 
*(Αν το έχεις σε διαφορετικό φάκελο στο Drive σου, άλλαξε το path παρακάτω)*

In [ ]:
%%bash
# ΠΡΟΣΟΧΗ: Άλλαξε το path του MyDrive ανάλογα με το πού έχεις το αρχείο!
cp "/content/drive/MyDrive/intersection_tox7_corr3.tsv" /content/missing_features.tsv

# Επιβεβαίωση ότι ήρθε
ls -l /content/missing_features.tsv

## 6. Εκτέλεση του Generation (Phase 4a)
Τρέχουμε το script. Βάζουμε `--ratio 1.0` για να επεξεργαστεί **και τα 318 features** (χωρίς τυχαία δειγματοληψία) και `--num_synthetic_samples 2` για να παράγει 2 ερωτήματα ανά feature.

In [ ]:
%%bash
cd /content/FAC-Synthesis/fac_synthesis/step1_contrastive_pair_construction/

python generate_data_llama_r1.py \
  --features /content/missing_features.tsv \
  --out /content/step1_queries \
  --ratio 1.0 \
  --num_synthetic_samples 2 \
  --temperature 0.8

## 7. Αντιγραφή Αποτελεσμάτων πίσω στο Drive
Μόλις τελειώσει, το παραγόμενο αρχείο θα λέγεται `step1_queries.queries.tsv`. Το στέλνουμε στο Drive για να μην χαθεί όταν κλείσει το Colab.

In [ ]:
%%bash
cp /content/FAC-Synthesis/fac_synthesis/step1_contrastive_pair_construction/step1_queries.queries.tsv /content/drive/MyDrive/step1_queries.queries.tsv
print("✅ Το αρχείο αποθηκεύτηκε στο Drive!")

---# Phase 4b: SAE-Scoring & Contrastive Pair ConstructionΕδώ ξεκινάει το δεύτερο μισό του Phase 4 (Αξιολόγηση των κειμένων που παρήγαγε το Llama).Για να γίνει αυτό, χρειαζόμαστε τα βάρη του SAE (τον "εγκέφαλο"). Κατεβάζουμε το έτοιμο SAE checkpoint.

In [ ]:
from huggingface_hub import hf_hub_download
import os
import shutil

SAE_REPO = "Zhongzhi1228/sae_llama_l16_h65536"
SAE_FILENAME = "TopK7_l16_h4096_epoch3.pth"

print(f"Κατέβασμα SAE από το {SAE_REPO}...")
sae_cache_path = hf_hub_download(repo_id=SAE_REPO, filename=SAE_FILENAME)

# Αντιγραφή στον φάκελο που το περιμένει το script
os.makedirs("/content/sae_weights", exist_ok=True)
shutil.copy(sae_cache_path, "/content/sae_weights/topk_l16_h65536.pth")
print("✅ Το SAE κατέβηκε και τοποθετήθηκε σωστά μεσω huggingface_hub!")


## 8. Το "Κρυφό" Βήμα: Сollect Spans στα Συνθετικά ΔεδομέναΠερνάμε τις 636 προτάσεις που φτιάξαμε μέσα από το Llama+SAE για να πάρουμε τα activations τους.

In [ ]:
%%bash
cd /content/FAC-Synthesis/sae_feature_analysis/interpret_features/

FAC_OUT_DIR=/content/out_synthetic python collect_spans.py 0 llama 0 1 \
    --data-path /content/FAC-Synthesis/fac_synthesis/step1_contrastive_pair_construction/step1_queries.queries.tsv \
    --threshold 0.0 \
    --sae-path /content/sae_weights/topk_l16_h65536.pth

## 9. Patch για το analyze_step1_synthetic_data.pyΟ κώδικας του paper έχει κενές μεταβλητές (paths) σε αυτό το αρχείο. Το κάνουμε patch για να βάλει τα σωστά paths.

In [ ]:
import os
analyze_path = "/content/FAC-Synthesis/fac_synthesis/step1_contrastive_pair_construction/analyze_step1_synthetic_data.py"

with open(analyze_path, "r") as f:
    code = f.read()

code = code.replace("FINAL_DECISION_FILE = ", "FINAL_DECISION_FILE = /content/missing_features.tsv")
code = code.replace("EXTSPANS_FILE = ", "TEXTSPANS_FILE = /content/out_synthetic/fac_saved_outputs/threshold_0.0/textspans_group0.tsv")
code = code.replace("YNTHETIC_QUERIES_FILE = ", "SYNTHETIC_QUERIES_FILE = /content/FAC-Synthesis/fac_synthesis/step1_contrastive_pair_construction/step1_queries.queries.tsv")
code = code.replace("OUTPUT_JSONL = ", "OUTPUT_JSONL = /content/step1_analyzed.jsonl")

with open(analyze_path, "w") as f:
    f.write(code)
print("✅ Το analyze_step1_synthetic_data.py είναι έτοιμο!")

## 10. Τρέχουμε το AnalyzeΘα διαβάσει τα σκορ από το σκανάρισμα και θα κρατήσει το Top-2 ανά feature.

In [ ]:
%%bash
cd /content/FAC-Synthesis/fac_synthesis/step1_contrastive_pair_construction/
python analyze_step1_synthetic_data.py

## 11. Patch & Τρέξιμο του merge_step1_failed_cases.pyΕδώ γίνεται η τελική δημιουργία των Contrastive Pairs (το αρχείο που πάει στο Round 2).

In [ ]:
merge_path = "/content/FAC-Synthesis/fac_synthesis/step1_contrastive_pair_construction/merge_step1_failed_cases.py"

with open(merge_path, "r") as f:
    code = f.read()

code = code.replace("FINAL_DECISION_FILE = \"xxx.tsv\"", "FINAL_DECISION_FILE = \"/content/missing_features.tsv\"")
code = code.replace("TRIPLETS_FILE = \"xxx.jsonl\"", "TRIPLETS_FILE = \"/content/step1_analyzed.jsonl\"")
code = code.replace("OUTPUT_FILE = \"xxx.jsonl\"", "OUTPUT_FILE = \"/content/step1_contrastive_pairs.jsonl\"")

with open(merge_path, "w") as f:
    f.write(code)

print("✅ Το merge script πατσαρίστηκε. Τρέχουμε...")

!cd /content/FAC-Synthesis/fac_synthesis/step1_contrastive_pair_construction/ && python merge_step1_failed_cases.py

## 12. Αποθήκευση στο Google DriveΣτέλνουμε το χρυσό αρχείο (jsonl) στο Drive για να το χρησιμοποιήσουμε στο Phase 4c (Round 2).

In [ ]:
%%bash
cp /content/step1_contrastive_pairs.jsonl /content/drive/MyDrive/
print("✅ ΤΕΛΟΣ! Το step1_contrastive_pairs.jsonl είναι στο Drive σου.")